# NB2 — Exploración interactiva de logs de auditoría

Carga un run completo (`.jsonl` o `.csv`) y visualiza la traza temporal de cada variable:
ruta activa, campo de obstáculos, eventos SLM, deadlocks, trayectoria 2D y latencias.

Equivalente Python del visor `viewer.html`, con acceso a todos los campos del log.

**Cómo usar:**
1. Ajustar `RUN_PATH` en §0 para apuntar a un directorio de corrida o a un `.jsonl` / `.csv`
2. Kernel → Run All Cells

## §0 — Selección de corrida

In [ ]:
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# ── CAMBIAR AQUÍ ──────────────────────────────────────────────────────────────
RUN_PATH = '../../airsim-runs/produccion/tier1/townsim_clear/slm/deep_vlm/seed_1'
# Puede ser:
#   - Directorio de corrida  →  se carga seed_*.csv y seed_*.jsonl automáticamente
#   - Ruta a .jsonl          →  se carga el JSONL directamente
#   - Ruta a .csv            →  se carga el CSV directamente
# ─────────────────────────────────────────────────────────────────────────────

ROUTE_COLORS = {
    'reactive':    '#4CAF50',
    'keep_going':  '#4CAF50',
    'evasive':     '#FF9800',
    'deliberative':'#2196F3',
    'girar_90':    '#F44336',
    'fsm':         '#9C27B0',
}

In [ ]:
def load_run(run_path: str) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    """Carga CSV (y opcionalmente JSONL) de un run. Devuelve (df_csv, df_jsonl)."""
    p = Path(run_path)

    # Resolver directorio → buscar seed_*.csv y seed_*.jsonl
    if p.is_dir():
        csvs  = sorted(p.glob('seed_*.csv'))
        jsonls = sorted(p.glob('seed_*.jsonl'))
        if not csvs:
            raise FileNotFoundError(f'No se encontró seed_*.csv en {p}')
        # Cargar el primero si hay varios; cambiar índice para seleccionar otro
        csv_path  = csvs[0]
        jsonl_path = jsonls[0] if jsonls else None
    elif p.suffix == '.csv':
        csv_path  = p
        jsonl_path = p.with_suffix('.jsonl') if p.with_suffix('.jsonl').exists() else None
    elif p.suffix == '.jsonl':
        jsonl_path = p
        csv_path  = p.with_suffix('.csv') if p.with_suffix('.csv').exists() else None
    else:
        raise ValueError(f'Ruta no reconocida: {run_path}')

    df_csv = pd.read_csv(csv_path) if csv_path and csv_path.exists() else None

    df_jsonl = None
    if jsonl_path and jsonl_path.exists():
        rows = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        df_jsonl = pd.json_normalize(rows)

    return df_csv, df_jsonl


df, df_j = load_run(RUN_PATH)

# Metadatos de la corrida
arm      = df['arm'].iloc[0] if df is not None else 'unknown'
scenario = df['scenario'].iloc[0] if df is not None else 'unknown'
seed     = int(df['seed'].iloc[0]) if df is not None else -1
n_cycles = len(df) if df is not None else 0
duration = df['t'].iloc[-1] - df['t'].iloc[0] if df is not None and n_cycles > 1 else 0
success_col = None  # el summary.json tiene success; el CSV no lo tiene directamente

print(f'Corrida: scenario={scenario}  arm={arm}  seed={seed}')
print(f'Ciclos: {n_cycles}  |  Duración: {duration:.1f} s')
if df is not None:
    print(f'Rutas presentes: {df["route"].value_counts().to_dict()}')

## §1 — Traza de route (nodo activo) a lo largo del tiempo

In [ ]:
fig, (ax_route, ax_dist) = plt.subplots(2, 1, figsize=(14, 5), sharex=True,
                                         gridspec_kw={'height_ratios': [2, 1]})

routes = df['route'].fillna('unknown')
times  = df['t'].values
unique_routes = routes.unique()

for i, (t, r) in enumerate(zip(times, routes)):
    color = ROUTE_COLORS.get(r, '#9E9E9E')
    ax_route.axvspan(t, times[i+1] if i+1 < len(times) else t+0.3,
                     facecolor=color, alpha=0.7, linewidth=0)

legend_patches = [mpatches.Patch(color=ROUTE_COLORS.get(r, '#9E9E9E'), label=r)
                  for r in unique_routes]
ax_route.legend(handles=legend_patches, loc='upper right', fontsize=8)
ax_route.set_ylabel('Ruta activa')
ax_route.set_yticks([])
ax_route.set_title(f'Traza de control — {arm}/{scenario}/seed={seed}')

ax_dist.plot(times, df['dist_to_wp_m'], color='#333', linewidth=1)
ax_dist.set_ylabel('dist_to_wp (m)')
ax_dist.set_xlabel('Tiempo (s)')

fig.tight_layout()
plt.show()

## §2 — ObstacleField en el tiempo (señal de percepción por sector)

In [ ]:
SECTORS = ['izquierda', 'centro', 'derecha']
SECTOR_COLORS = {'izquierda': '#2196F3', 'centro': '#F44336', 'derecha': '#FF9800'}

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
metrics = ['occ', 'ttc_s', 'conf', 'blocked']
titles  = ['Occupancy', 'TTC (s)', 'Confidence', 'Blocked']

for ax, metric, title in zip(axes, metrics, titles):
    for sector in SECTORS:
        col = f'field_{sector}_{metric}'
        if col in df.columns:
            y = pd.to_numeric(df[col], errors='coerce')
            ax.plot(df['t'], y, label=sector, color=SECTOR_COLORS[sector],
                    linewidth=1, alpha=0.85)
    if metric == 'blocked':
        # Marcar ciclos con centro bloqueado
        blocked = df['field_centro_blocked'].astype(str).str.lower() == 'true'
        ax.axhspan(0, 1, where=blocked.values, color='red', alpha=0.15)
    ax.set_ylabel(title)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Tiempo (s)')
fig.suptitle('ObstacleField por sector — señal de percepción', fontsize=11)
fig.tight_layout()
plt.show()

## §3 — Eventos SLM / deliberación

In [ ]:
# Ciclos donde el SLM fue invocado
slm_mask = df['slm_invoked'].astype(str).str.lower() == 'true'
df_slm = df[slm_mask].copy()
print(f'Invocaciones SLM: {len(df_slm)}')

if not df_slm.empty:
    # Mostrar prompt y respuesta (truncados) de cada invocación
    for _, row in df_slm.iterrows():
        prompt = str(row.get('slm_prompt', ''))[:120]
        resp   = str(row.get('slm_raw_response', ''))[:80]
        adherent = row.get('slm_adherent', '?')
        lat = row.get('slm_latency_ms', '?')
        print(f'  t={row["t"]:.1f}s  cycle={int(row["cycle"])}  adherent={adherent}  lat={lat}ms')
        print(f'    prompt:   {prompt}...')
        print(f'    response: {resp}...')
        print()

In [ ]:
# Timeline + distribución de latencia SLM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Timeline de invocaciones
ax1.plot(df['t'], df['route'].map(lambda r: 1 if r == 'deliberative' else 0),
         color='#2196F3', linewidth=1, alpha=0.5, label='deliberative')
if not df_slm.empty:
    ax1.vlines(df_slm['t'], 0, 1, color='#F44336', linewidth=1.2, label='SLM invoked')
ax1.set_xlabel('Tiempo (s)')
ax1.set_title('Timeline de invocaciones SLM')
ax1.legend(fontsize=8)

# Histograma de latencia SLM
if not df_slm.empty and 'slm_latency_ms' in df_slm.columns:
    lats = pd.to_numeric(df_slm['slm_latency_ms'], errors='coerce').dropna()
    if not lats.empty:
        ax2.hist(lats, bins=20, color='#2196F3', alpha=0.8, edgecolor='white')
        p50 = lats.quantile(0.50)
        p95 = lats.quantile(0.95)
        ax2.axvline(p50, color='green', linestyle='--', label=f'p50={p50:.0f}ms')
        ax2.axvline(p95, color='red', linestyle='--', label=f'p95={p95:.0f}ms')
        ax2.set_xlabel('Latencia SLM (ms)')
        ax2.set_ylabel('Frecuencia')
        ax2.set_title('Distribución de latencia SLM')
        ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'Sin invocaciones SLM', ha='center', va='center', transform=ax2.transAxes)

fig.tight_layout()
plt.show()

## §4 — Eventos de deadlock y resolución

In [ ]:
# Detectar transiciones a girar_90 (indicador de deadlock activo)
route_series = df['route'].fillna('unknown')
girar_mask = route_series == 'girar_90'
transitions = (girar_mask & ~girar_mask.shift(1, fill_value=False))

deadlock_events = []
i = 0
idxs = df.index.tolist()
while i < len(idxs):
    idx = idxs[i]
    if girar_mask.iloc[i]:
        start_i = i
        while i < len(idxs) and girar_mask.iloc[i]:
            i += 1
        end_i = i - 1
        t_start = df['t'].iloc[start_i]
        t_end   = df['t'].iloc[end_i]
        cycle_start = int(df['cycle'].iloc[start_i])
        cycle_end   = int(df['cycle'].iloc[end_i])
        resolved = i < len(idxs) and route_series.iloc[i] != 'girar_90'
        deadlock_events.append({
            'ciclo_inicio': cycle_start,
            'ciclo_fin':    cycle_end,
            'duración_ciclos': cycle_end - cycle_start + 1,
            't_inicio': t_start,
            't_fin':    t_end,
            'resuelto': resolved,
        })
    else:
        i += 1

if deadlock_events:
    df_deadlocks = pd.DataFrame(deadlock_events)
    print(f'Eventos deadlock detectados: {len(df_deadlocks)}')
    display(df_deadlocks)
else:
    print('Sin eventos deadlock (girar_90) en esta corrida.')

## §5 — Trayectoria 2D del dron

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

routes_in_run = df['route'].fillna('unknown').unique()
for route in routes_in_run:
    mask = df['route'] == route
    color = ROUTE_COLORS.get(route, '#9E9E9E')
    ax.scatter(df.loc[mask, 'pos_x'], df.loc[mask, 'pos_y'],
               c=color, s=4, alpha=0.5, label=route)

# Marcar inicio y fin
ax.plot(df['pos_x'].iloc[0],  df['pos_y'].iloc[0],  'k^', markersize=10, label='Inicio')
ax.plot(df['pos_x'].iloc[-1], df['pos_y'].iloc[-1], 'k*', markersize=12, label='Fin')

# Marcar posiciones de deadlock
if deadlock_events:
    for ev in deadlock_events:
        mask_dl = (df['cycle'] >= ev['ciclo_inicio']) & (df['cycle'] <= ev['ciclo_fin'])
        ax.scatter(df.loc[mask_dl, 'pos_x'], df.loc[mask_dl, 'pos_y'],
                   marker='x', c='black', s=40, linewidths=1.5)

ax.set_xlabel('pos_x (m)')
ax.set_ylabel('pos_y (m)')
ax.set_title(f'Trayectoria 2D — {arm}/{scenario}/seed={seed}  (× = deadlock)')
ax.legend(fontsize=8, markerscale=2)
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

## §6 — Latencia del grafo por ciclo

In [ ]:
# Parsear latency_ms_json (columna CSV = JSON string)
def extract_graph_latency(val) -> float:
    try:
        d = json.loads(str(val))
        return float(d.get('graph', float('nan')))
    except Exception:
        return float('nan')


df['latency_graph_ms'] = df['latency_ms_json'].map(extract_graph_latency)
p50 = df['latency_graph_ms'].quantile(0.50)
p95 = df['latency_graph_ms'].quantile(0.95)
print(f'Latencia grafo: p50={p50:.1f}ms  p95={p95:.1f}ms')

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df['t'], df['latency_graph_ms'], linewidth=0.8, color='#333', alpha=0.7)

# Marcar picos (> p95)
picos = df['latency_graph_ms'] > p95
ax.scatter(df.loc[picos, 't'], df.loc[picos, 'latency_graph_ms'],
           color='red', s=8, zorder=5, label=f'Picos >p95 ({p95:.0f}ms)')
ax.axhline(p50, color='green', linestyle='--', linewidth=0.8, label=f'p50={p50:.0f}ms')
ax.axhline(p95, color='red',   linestyle='--', linewidth=0.8, label=f'p95={p95:.0f}ms')

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('Latencia grafo (ms)')
ax.set_title('Latencia del grafo por ciclo')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## §7 — Comparar dos corridas (misma semilla, brazos distintos)

In [ ]:
# ── CAMBIAR AQUÍ ──────────────────────────────────────────────────────────────
RUN_A = '../../airsim-runs/produccion/tier1/townsim_clear/slm/deep_vlm/seed_1'
RUN_B = '../../airsim-runs/produccion/tier1/townsim_clear/reactive/deep_vlm/seed_1'
# ─────────────────────────────────────────────────────────────────────────────

try:
    df_a, _ = load_run(RUN_A)
    df_b, _ = load_run(RUN_B)
    df_a['latency_graph_ms'] = df_a['latency_ms_json'].map(extract_graph_latency)
    df_b['latency_graph_ms'] = df_b['latency_ms_json'].map(extract_graph_latency)

    arm_a = df_a['arm'].iloc[0]
    arm_b = df_b['arm'].iloc[0]
    print(f'Corrida A: {arm_a} — {df_a["t"].iloc[-1]:.1f}s  |  Corrida B: {arm_b} — {df_b["t"].iloc[-1]:.1f}s')

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)

    for ax, df_run, arm_label, color in [
        (axes[0], df_a, arm_a, '#2196F3'),
        (axes[1], df_b, arm_b, '#4CAF50'),
    ]:
        routes = df_run['route'].fillna('unknown')
        times  = df_run['t'].values
        for i, (t, r) in enumerate(zip(times, routes)):
            c = ROUTE_COLORS.get(r, '#9E9E9E')
            ax.axvspan(t, times[i+1] if i+1 < len(times) else t+0.3, facecolor=c, alpha=0.7, linewidth=0)
        ax.set_title(f'Route — {arm_label}  (duración: {times[-1]:.1f}s)')
        ax.set_yticks([])

    axes[2].plot(df_a['t'], df_a['dist_to_wp_m'], label=arm_a, color='#2196F3', linewidth=1)
    axes[2].plot(df_b['t'], df_b['dist_to_wp_m'], label=arm_b, color='#4CAF50', linewidth=1)
    axes[2].set_xlabel('Tiempo (s)')
    axes[2].set_ylabel('dist_to_wp (m)')
    axes[2].legend(fontsize=8)

    fig.suptitle(f'Comparación {arm_a} vs {arm_b} — misma semilla', fontsize=11)
    fig.tight_layout()
    plt.show()

except Exception as e:
    print(f'No se pudo cargar la comparación: {e}')
    print('Ajustar RUN_A y RUN_B para apuntar a corridas válidas del mismo escenario/semilla.')